In [11]:
%run 3-Pipeline.ipynb

In [12]:
from pathlib import Path
import joblib
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [13]:
MODEL_DIR = Path("Model_save")

RANDOM_STATE = 42

In [ ]:
model = Pipeline([
    ("preprocessing", pipeline),
    ("classifier", GradientBoostingClassifier(
     random_state=RANDOM_STATE
    )),
])

Valeurs restent par dfaut n_estimators et max_depth

subsample: permet de tirer un échantillon aléatoire à chaque arbre 

min sample leaf: définit le nombre nimimale d individus possibles à chaque feuille de l arbre, plus il est faible plus le modèle crées des règles précises

In [15]:
param_grid = {
    "classifier__learning_rate": [0.01, 0.05, 0.1],
    "classifier__subsample": [0.7, 0.85, 1.0],
    "classifier__min_samples_leaf": [1, 5, 10],
}

# Validation croisée stratifiée à 5 plis, uniquement sur la partie train
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

search = GridSearchCV(model, param_grid, cv=cv, scoring="f1", n_jobs=-1)
search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__learning_rate': [0.01, 0.05, ...], 'classifier__min_samples_leaf': [1, 5, ...], 'classifier__subsample': [0.7, 0.85, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 

In [16]:
search.best_params_, search.best_score_

({'classifier__learning_rate': 0.1,
  'classifier__min_samples_leaf': 10,
  'classifier__subsample': 0.85},
 np.float64(0.7683394255773914))

In [17]:
model = search.best_estimator_

In [18]:
def calculer_metriques_clf(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
    }

Seuil de décision conservé à 0,5. Testé de 0,30 à 0,60 en validation croisée sur le jeu d'entraînement, baisser le seuil permet au modèle de repérer plus de survivants mais se trompe plus souvent quand il en annonce un, l augmenter permet de moins se tromper lors de l annonce d'un survivant mais il en manque d'avantage => valeur par défaut
 

In [19]:
SEUIL = 0.5

proba = model.predict_proba(X_test)[:, 1]
y_pred = (proba >= SEUIL).astype(int)

calculer_metriques_clf(y_test, y_pred)

{'accuracy': 0.8324022346368715,
 'precision': 0.8305084745762712,
 'recall': 0.7101449275362319,
 'f1': 0.765625}

In [20]:
joblib.dump(model, MODEL_DIR / "gradient_boosting.joblib")

['Model_save/gradient_boosting.joblib']